# Experiment 17 — Theme Description + Stage — Individual Calls, No Record Key in Prompt

One LLM call per Stage. The model receives Theme context, Stage data, and candidate L3 capabilities only. No source record key is included in the model prompt.

## What the LLM sees

- Theme Description
- Stage data: stage_id, stage_name, stage_description, entrance_criteria, exit_criteria
- Candidate L3: capability_id, capability_name, capability_description, capability_tier

**Not sent:** source record key, record description/success criteria, ground truth, L1/L2 hierarchy.

## Configuration and imports

In [ ]:
from pathlib import Path
from time import perf_counter
import ast
import json
import os

import pandas as pd
from IPython.display import display

from common import (
    call_llm_with_metrics,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
)

NOTEBOOK_DIR = Path.cwd()


def resolve_data_path(relative_path: str, env_var: str) -> Path:
    override = os.getenv(env_var)
    if override:
        return Path(override).expanduser()

    relative_path = Path(relative_path)
    search_roots = [
        NOTEBOOK_DIR,
        NOTEBOOK_DIR.parent,
        NOTEBOOK_DIR / "l3_experiments",
    ]
    attempted = []
    seen = set()

    for root in search_roots:
        candidate = root / relative_path
        key = str(candidate.resolve(strict=False))
        if key in seen:
            continue
        seen.add(key)
        attempted.append(candidate)
        if candidate.exists():
            return candidate

    tried = "\n  - ".join(str(path) for path in attempted)
    raise FileNotFoundError(
        f"Could not find {relative_path}. Tried:\n  - {tried}"
    )


PARQUET_PATH = resolve_data_path("full_golden.parquet", "L3_FULL_GOLDEN_PATH")
STAGE_PATH = resolve_data_path("VSSrv.csv", "L3_STAGE_PATH")
STAGE_CAPABILITY_MAP_PATH = resolve_data_path(
    "VSSCaprv (1).csv",
    "L3_STAGE_CAPABILITY_MAP_PATH",
)
GROUND_TRUTH_PATH = resolve_data_path(
    "results/epic_l3_ground_truth_full_golden.xlsx",
    "L3_GROUND_TRUTH_PATH",
)

SAMPLE_SIZE = 50
SAMPLE_SEED = 42

EXPERIMENT_NAME = "E17_THEME_DESCRIPTION_STAGE_INDIVIDUAL_NO_RECORD_KEY"


## Retrieval and candidate construction

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def parse_list_value(value) -> list[str]:
    if value is None:
        return []

    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()

    if isinstance(value, (list, tuple, set)):
        return [clean_text(item) for item in value if clean_text(item)]

    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]

    if isinstance(parsed, (list, tuple, set)):
        return [clean_text(item) for item in parsed if clean_text(item)]
    return [clean_text(parsed)] if clean_text(parsed) else []


def read_table(path, *, sheet_name=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(
            path,
            dtype=str,
            encoding="cp1252",
            encoding_errors="replace",
        )

    return pd.read_excel(path, sheet_name=sheet_name, dtype=str)


def load_evaluation_population():
    population = read_table(
        GROUND_TRUTH_PATH,
        sheet_name="evaluation_population",
    )

    required = {
        "theme_key",
        "epic_key",
        "stage_ids",
        "gt_l3_ids",
        "candidate_l3_ids",
    }
    missing = required.difference(population.columns)
    if missing:
        raise KeyError(
            f"evaluation_population is missing columns: {sorted(missing)}"
        )

    population = (
        population
        .drop_duplicates(subset=["theme_key", "epic_key"], keep="first")
        .sort_values(["theme_key", "epic_key"], kind="stable")
        .reset_index(drop=True)
    )

    if len(population) < SAMPLE_SIZE:
        raise ValueError(
            f"Need {SAMPLE_SIZE} valid records, found only {len(population)}."
        )

    sample = population.sample(
        n=SAMPLE_SIZE,
        random_state=SAMPLE_SEED,
        replace=False,
    )

    return sample.sort_values(
        ["theme_key", "epic_key"],
        kind="stable",
    ).reset_index(drop=True)


evaluation_population = load_evaluation_population()

selected_pairs = set(
    zip(
        evaluation_population["theme_key"],
        evaluation_population["epic_key"],
    )
)
selected_theme_ids = set(evaluation_population["theme_key"])


def load_themes():
    frame = read_table(PARQUET_PATH)

    required = {"key", "description", "businessNeeds", "epic_keys"}
    missing = required.difference(frame.columns)
    if missing:
        raise KeyError(
            f"full_golden.parquet is missing columns: {sorted(missing)}"
        )

    themes = {}
    found_pairs = set()

    for _, row in frame.iterrows():
        theme_id = clean_text(row.get("key"))
        if theme_id not in selected_theme_ids:
            continue

        selected_records = []
        for source_key in parse_list_value(row.get("epic_keys")):
            if (theme_id, source_key) in selected_pairs:
                selected_records.append(source_key)
                found_pairs.add((theme_id, source_key))

        if not selected_records:
            continue

        themes[theme_id] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
            "record_keys": selected_records,
        }

    missing_pairs = selected_pairs - found_pairs
    if missing_pairs:
        raise ValueError(
            "Selected Theme/source-key pairs were not found in full_golden.parquet: "
            f"{sorted(missing_pairs)}"
        )

    return themes


themes = load_themes()
stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)


def stage_context(stage_id):
    match = stage_frame.loc[
        stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id
    ]

    if match.empty:
        raise KeyError(f"No stage metadata for {stage_id}")

    row = match.iloc[0]

    return {
        "stage_id": stage_id,
        "stage_name": clean_text(row["Value Stream Stage Name"]),
        "stage_description": clean_text(row["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"]),
    }


def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()

    rows = (
        rows
        .drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean_text(row["Capability ID"]),
            "capability_name": clean_text(row["Capability Name"]),
            "capability_description": clean_text(row["Capability Description"]),
            "capability_tier": clean_text(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


print(
    f"Selected {len(evaluation_population)} valid records "
    f"with seed={SAMPLE_SEED} across {len(themes)} Themes."
)
display(evaluation_population.head(50))


## Production prompt

In [ ]:
SYSTEM_PROMPT = """You are performing Level 3 business capability classification.

An L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.

Use the Theme Description as the business context.
Use the supplied Value Stream Stage data to determine the relevant business boundary.
Select the candidate L3 capabilities that are directly supported by that context.

EVIDENCE

Theme Description explains the scope, intent, and business activity being addressed.

Stage data identifies the part of the business process relevant to this classification.

For each candidate L3:
- capability_id is the exact identifier to return when selected.
- capability_description is the primary semantic definition of the business function.
- capability_name is a supporting business label.
- capability_tier is supporting taxonomy context only.

Do not infer business meaning from capability_id.

CLASSIFICATION

1. Determine the business functions expressed by the Theme Description.
2. Constrain those functions using the supplied Stage data.
3. Compare that evidence against the candidate L3 capability descriptions.
4. Select every candidate whose business function is directly supported by the supplied context.

Do not select a capability merely because:
- it belongs to the supplied Stage,
- it shares terminology with the context,
- it is broadly related,
- it is upstream or downstream,
- it commonly supports another capability.

Only return capability_id values supplied in candidate_l3_capabilities.

If no candidate is supported, return an empty list.

OUTPUT

Return JSON only:

{"l3":["CAP00000123","CAP00000456"]}

Do not return reasons, explanations, Markdown, or additional fields."""


def build_user_prompt(theme, stage, candidate_rows):
    payload = {
        "theme_description": theme["theme_description"],
        "stage_data": stage,
        "candidate_l3_capabilities": candidate_rows,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prompt preview — printed exactly as sent

In [ ]:
preview_row = evaluation_population.iloc[0]
preview_theme = themes[clean_text(preview_row["theme_key"])]
preview_stage_id = parse_list_value(preview_row["stage_ids"])[0]
preview_stage = stage_context(preview_stage_id)
preview_candidates = candidate_rows_for_stage(preview_stage_id)
preview_user_prompt = build_user_prompt(
    preview_theme,
    preview_stage,
    preview_candidates,
)

print("SYSTEM PROMPT")
print("=" * 80)
print(SYSTEM_PROMPT)
print()
print("FORMATTED USER PROMPT")
print("=" * 80)
print(preview_user_prompt)


## Prediction and evaluation

In [ ]:
def validate_l3_response(payload, candidate_ids):
    if not isinstance(payload, dict) or set(payload) != {"l3"}:
        raise ValueError("Response must contain exactly one top-level field: l3.")

    raw = payload["l3"]
    if not isinstance(raw, list):
        raise ValueError("l3 must be a list.")

    allowed = set(candidate_ids)
    selected = []
    seen = set()

    for capability_id in raw:
        if not isinstance(capability_id, str):
            raise ValueError("Every L3 selection must be a capability_id string.")

        capability_id = capability_id.strip()

        if capability_id not in allowed:
            raise ValueError(
                f"{capability_id} is not in the supplied candidate list."
            )

        if capability_id in seen:
            raise ValueError(f"Duplicate capability_id: {capability_id}")

        seen.add(capability_id)
        selected.append(capability_id)

    return selected


def predict_stage(gateway, theme, stage_id):
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, stage, candidates)

    if not candidates:
        return {
            "selected": [],
            "user_prompt": user_prompt,
            "raw_response": None,
            "metrics": None,
        }

    raw_response, metrics = call_llm_with_metrics(
        gateway,
        SYSTEM_PROMPT,
        user_prompt,
        id="9zdn8n",
        reasoning_effort="low",
    )

    selected = validate_l3_response(
        parse_json_response(raw_response),
        [candidate["capability_id"] for candidate in candidates],
    )

    return {
        "selected": selected,
        "user_prompt": user_prompt,
        "raw_response": raw_response,
        "metrics": metrics,
    }


def run_experiment():
    gateway = load_gateway()
    result_rows = []
    call_rows = []

    for row in evaluation_population.to_dict(orient="records"):
        theme_id = clean_text(row["theme_key"])
        source_key = clean_text(row["epic_key"])
        stage_ids = parse_list_value(row["stage_ids"])
        truth = parse_list_value(row["gt_l3_ids"])
        theme = themes[theme_id]

        predicted = set()
        status = "ok"
        error = None

        for stage_id in stage_ids:
            started = perf_counter()

            try:
                result = predict_stage(gateway, theme, stage_id)

                if result["metrics"] is not None:
                    metrics = result["metrics"]
                    call_rows.append({
                        "experiment": EXPERIMENT_NAME,
                        "theme_id": theme_id,
                        "record_key": source_key,
                        "stage_id": stage_id,
                        "status": "ok",
                        "latency_seconds": metrics.get("latency_seconds"),
                        "input_tokens": metrics.get("input_tokens"),
                        "output_tokens": metrics.get("output_tokens"),
                        "total_tokens": metrics.get("total_tokens"),
                        "error": None,
                    })

                predicted.update(result["selected"])

            except Exception as exc:
                status = "error"
                error = str(exc)

                call_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "record_key": source_key,
                    "stage_id": stage_id,
                    "status": "error",
                    "latency_seconds": perf_counter() - started,
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "error": error,
                })
                break

        scores = (
            score_sets(predicted, truth)
            if status == "ok"
            else {
                "exact_match": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "predicted_count": len(predicted),
                "truth_count": len(truth),
            }
        )

        result_rows.append({
            "experiment": EXPERIMENT_NAME,
            "theme_id": theme_id,
            "record_key": source_key,
            "stage_ids": stage_ids,
            "predicted_l3_ids": sorted(predicted),
            "gt_l3_ids": truth,
            "status": status,
            "error": error,
            **scores,
        })

    return pd.DataFrame(result_rows), pd.DataFrame(call_rows)


results, call_metrics = run_experiment()

scored = results.loc[results["status"].eq("ok")].copy()
successful_calls = call_metrics.loc[call_metrics["status"].eq("ok")].copy()

summary = pd.DataFrame([{
    "scope": "fixed_50_valid_records_seed_42_individual_stage_calls",
    "evaluated_records": len(scored),
    "exact_match_accuracy": scored["exact_match"].mean() if len(scored) else 0.0,
    "mean_precision": scored["precision"].mean() if len(scored) else 0.0,
    "mean_recall": scored["recall"].mean() if len(scored) else 0.0,
    "mean_f1": scored["f1"].mean() if len(scored) else 0.0,
}])

latency_tokens = pd.DataFrame([{
    "successful_calls": len(successful_calls),
    "failed_calls": int(call_metrics["status"].eq("error").sum()) if len(call_metrics) else 0,
    "avg_latency_seconds": successful_calls["latency_seconds"].mean() if len(successful_calls) else None,
    "p50_latency_seconds": successful_calls["latency_seconds"].quantile(0.50) if len(successful_calls) else None,
    "p95_latency_seconds": successful_calls["latency_seconds"].quantile(0.95) if len(successful_calls) else None,
    "avg_input_tokens": successful_calls["input_tokens"].mean() if len(successful_calls) else None,
    "avg_output_tokens": successful_calls["output_tokens"].mean() if len(successful_calls) else None,
    "avg_total_tokens": successful_calls["total_tokens"].mean() if len(successful_calls) else None,
    "total_input_tokens": successful_calls["input_tokens"].sum() if len(successful_calls) else 0,
    "total_output_tokens": successful_calls["output_tokens"].sum() if len(successful_calls) else 0,
    "total_tokens": successful_calls["total_tokens"].sum() if len(successful_calls) else 0,
    "tokens_per_scored_record": (
        successful_calls["total_tokens"].sum() / len(scored)
        if len(scored) else None
    ),
}])

print("Evaluation summary")
display(summary)
print("LLM latency / token summary")
display(latency_tokens)
print("Per-call metrics")
display(call_metrics.head(100))
print("Results")
display(results.head(50))

output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    "results",
    extra_sheets={
        "evaluation_summary": summary,
        "llm_metrics": call_metrics,
        "latency_tokens": latency_tokens,
        "evaluation_population": evaluation_population,
    },
)
print(f"Saved {output_path}")
